In [1]:
!pip install kagglehub
import os
import random
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import datasets, transforms, models

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("debeshjha1/surgical-instrument-classification")
print("Downloaded dataset path:", path)

/Users/abroadhub/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Downloaded dataset path: /Users/abroadhub/.cache/kagglehub/datasets/debeshjha1/surgical-instrument-classification/versions/1


In [3]:
root_path = Path(path)

for item in root_path.iterdir():
    print(item)

/Users/abroadhub/.cache/kagglehub/datasets/debeshjha1/surgical-instrument-classification/versions/1/Instrument Data


In [4]:
from pathlib import Path

data_dir = Path(path) / "Instrument Data"

print(data_dir)

/Users/abroadhub/.cache/kagglehub/datasets/debeshjha1/surgical-instrument-classification/versions/1/Instrument Data


In [5]:
for item in data_dir.iterdir():
    print(item.name)

Clip
Chain
No Instrument
White Tube
Doubt
Hook
Hold


In [6]:
device = torch.device( "mps" if torch.backends.mps.is_available() else "cpu")
print(device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

mps


In [7]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [8]:
full_dataset = datasets.ImageFolder(root=data_dir)

print("Classes:", full_dataset.classes)
print("Class to index:", full_dataset.class_to_idx)
print("Total images:", len(full_dataset))

Classes: ['Chain', 'Clip', 'Doubt', 'Hold', 'Hook', 'No Instrument', 'White Tube']
Class to index: {'Chain': 0, 'Clip': 1, 'Doubt': 2, 'Hold': 3, 'Hook': 4, 'No Instrument': 5, 'White Tube': 6}
Total images: 1315


In [9]:
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 920
Val: 197
Test: 198


In [10]:
class CustomSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label   

In [11]:
train_dataset = CustomSubset(train_dataset, transform=train_transform)
val_dataset = CustomSubset(val_dataset, transform=test_transform)
test_dataset = CustomSubset(test_dataset, transform=test_transform)

In [12]:
# labels from original split indices
train_indices = train_dataset.subset.indices
train_labels = [full_dataset.samples[i][1] for i in train_indices]

class_count = np.bincount(train_labels)
print("Class counts in train split:", class_count)

class_weights = 1. / class_count
sample_weights = [class_weights[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

Class counts in train split: [ 88  32  17  23 120  12 628]


In [13]:
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [14]:
images, labels = next(iter(train_loader))
print("Image batch shape:", images.shape)   # [batch_size, 3, 224, 224]
print("Label batch shape:", labels.shape)   # [batch_size]

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


In [17]:
import torchvision.models as models
import torch.nn as nn

import torchvision.models as models
resnet18 = models.resnet18(pretrained=True)

# freeze all layers
for param in resnet18.parameters():
    param.requires_grad = False

# replace final layer
num_classes = len(full_dataset.classes)
resnet18.fc = nn.Linear(resnet18.fc.in_features, num_classes)

# make fc trainable
for param in resnet18.fc.parameters():
    param.requires_grad = True

resnet18 = resnet18.to(device)

In [20]:
learning_rate = 0.0001
epochs = 15
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet18.fc.parameters(), lr=learning_rate)

# training loop
for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = resnet18(batch_features)

    loss = criterion(outputs, batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 1.6599133014678955
Epoch: 2 , Loss: 1.616654367282473
Epoch: 3 , Loss: 1.5946204621216347
Epoch: 4 , Loss: 1.5576377572684452
Epoch: 5 , Loss: 1.5270798699609165
Epoch: 6 , Loss: 1.532798487564613
Epoch: 7 , Loss: 1.5016334509027416
Epoch: 8 , Loss: 1.4774787261568267
Epoch: 9 , Loss: 1.4515078643272663
Epoch: 10 , Loss: 1.4152689432275707
Epoch: 11 , Loss: 1.4464887627239884
Epoch: 12 , Loss: 1.3744385489102067
Epoch: 13 , Loss: 1.352436817925552
Epoch: 14 , Loss: 1.3804366054206059
Epoch: 15 , Loss: 1.3423839560870467


In [21]:
# evaluate the TEST data
total = 0
correct = 0
with torch.no_grad():

  for batch_features, batch_labels in test_loader:

    # move data to gpu
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = resnet18(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.2828282828282828
